# 10. Data Exchange and Interoperability

Export a labelled room to JSON, then import it and check its geometry and attributes.

**Goal:** Complete one small workflow. Run the cells from top to bottom in a fresh kernel.

**Extra dependencies:** None beyond the base environment.

All examples use metres unless stated otherwise.


In [ ]:
import os
os.environ["TOPOLOGICPY_CORE_BACKEND"] = "pythonocc"

import topologicpy
from topologicpy.Core import Core

assert topologicpy.__version__ == "0.9.86", "Use TopologicPy 0.9.86 for these tutorials."
assert type(Core.Backend()).__name__ == "PythonOCCBackend", "Restart with the pythonocc backend."
print("TopologicPy", topologicpy.__version__, "—", type(Core.Backend()).__name__)


## Write a small model

Use a relative output folder. The JSON representation is a convenient TopologicPy round trip, not a universal exchange standard.


In [ ]:
from pathlib import Path
from topologicpy.Cell import Cell
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

room = Cell.Prism(width=4, length=3, height=2)
room = Topology.SetDictionary(room, Dictionary.ByKeysValues(["name"], ["Studio"]))
text = Topology.JSONString([room])
output = Path("results")
output.mkdir(exist_ok=True)
path = output / "10_room.json"
path.write_text(text, encoding="utf-8")
print("Saved:", path)


## Read and verify

Import returns a list of topologies. Check both the volume and the semantic data after the round trip.


In [ ]:
restored = Topology.ByJSONString(path.read_text(encoding="utf-8"))
restored_room = restored[0]
name = Dictionary.ValueAtKey(Topology.Dictionary(restored_room), "name")
print("Restored:", name, Cell.Volume(restored_room))
assert abs(Cell.Volume(restored_room) - 24) < 0.0001
assert name == "Studio"


## Try it yourself

Add an occupancy property and verify it survives the round trip.
